# ChronoPDE V2 Phase 6A — sealed confirmatory generation

Attach the private dataset containing only `chronopde_v2_phase5_outputs.zip`, enable Internet, and use a CPU session. This generates the 256 identities frozen in Phase 2; it performs no model evaluation.

In [ ]:
import shutil
import subprocess
import sys
from pathlib import Path

URL = "https://github.com/madhavkapoor13/ChronoPDE.git"
BRANCH = "codex/chronopde-v2-phase6"
REPOSITORY = Path("/kaggle/working/ChronoPDE")
if not REPOSITORY.exists():
    subprocess.run(
        ["git", "clone", "--branch", BRANCH, "--single-branch", URL, str(REPOSITORY)], check=True
    )
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", "-e", str(REPOSITORY)], check=True
)
archives = list(Path("/kaggle/input").rglob("chronopde_v2_phase5_outputs.zip"))
assert len(archives) == 1, f"Expected one curated Phase 5 ZIP, found: {archives}"
PHASE5 = archives[0]
print("Phase 5:", PHASE5)

In [ ]:
command = [
    sys.executable,
    "scripts/chronopde_v2.py",
    "phase6",
    "generate",
    "--phase5-archive",
    str(PHASE5),
    "--format",
    "json",
]
result = subprocess.run(command, cwd=REPOSITORY)
run_roots = list(
    (REPOSITORY / "artifacts/chronopde_v2/runs").glob(
        "chronopde_v2-p6-reaction_diffusion-data-reference-confirmatory-*"
    )
)
assert len(run_roots) == 1, run_roots
run_root = run_roots[0]
for name in (
    "chronopde_v2_confirmatory.h5",
    "chronopde_v2_confirmatory.h5.sha256",
    "chronopde_v2_phase6_generation_evidence.zip",
):
    source = run_root / name
    if source.is_file():
        shutil.copy2(source, Path("/kaggle/working") / name)
print("Generation return code:", result.returncode)
print("Download both Phase 6 generation outputs from the Output tab.")
assert result.returncode == 0, (
    "Generation failed; preserve the notebook output/cache before retrying."
)